# Implementación de Transformers para Procesamiento de Lenguaje Natural (NLP)
### Chatbot conversacional con el dataset DailyDialog

**Objetivo:** implementar un modelo basado en arquitecturas de Transformers (encoder-decoder con atención multi-cabezal) capaz de responder a un mensaje escrito por el usuario, comportándose como un chatbot. Se utiliza el dataset **DailyDialog** (diálogos cotidianos) usando `datasets` de HuggingFace.

Al final se evalúa con la métrica **BLEU** y se incluye una función para chatear con el modelo.

## 1. Carga y Exploración del Dataset: DailyDialog

In [ ]:
# Instalar dependencias (Colab ya incluye TensorFlow)
!pip -q install datasets

from datasets import load_dataset
import numpy as np
import pandas as pd

# Cargar la versión Parquet del dataset (no requiere trust_remote_code)
dataset = load_dataset("OpenRL/daily_dialog")

print("--- TAMAÑO DEL DATASET ---")
print(f"Entrenamiento: {dataset['train'].num_rows} diálogos")
print(f"Validación:    {dataset['validation'].num_rows} diálogos")
print(f"Prueba:        {dataset['test'].num_rows} diálogos")

ejemplo = dataset['train'][0]
print("\n--- EJEMPLO DE DIÁLOGO ---")
for i, frase in enumerate(ejemplo['dialog']):
    print(f"  Turno {i}: {frase.strip()}")

In [ ]:
# Construir pares (mensaje -> respuesta) a partir de turnos consecutivos
# Cada par es (entrada, salida) para entrenar el chatbot.
def construir_pares(split):
    entradas, respuestas = [], []
    for dialogo in dataset[split]['dialog']:
        for i in range(len(dialogo) - 1):
            entradas.append(dialogo[i].strip())
            respuestas.append(dialogo[i + 1].strip())
    return entradas, respuestas

train_in, train_out = construir_pares('train')
test_in,  test_out  = construir_pares('test')

print(f"Pares de entrenamiento: {len(train_in)}")
print(f"Pares de prueba:        {len(test_in)}")

print("\nEjemplos de pares (mensaje -> respuesta):")
for i in range(3):
    print(f"  IN : {train_in[i]}")
    print(f"  OUT: {train_out[i]}\n")

## 2. Implementación del Modelo Transformer

Se construye un **encoder-decoder** con:
- *Embedding* + codificación posicional.
- *Multi-Head Attention* en encoder (auto-atención) y decoder (auto-atención causal + atención cruzada sobre el encoder).
- Salida con capa densa sobre el vocabulario para predecir el siguiente token.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Embedding, MultiHeadAttention,
                                      LayerNormalization, Dropout, Layer)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------- Hiperparámetros ----------------
VOCAB_SIZE    = 8000      # tamaño del vocabulario
MAX_LEN       = 40        # longitud máxima de secuencia
EMBED_DIM     = 128       # dimensión del embedding (d_model)
NUM_HEADS     = 4         # cabezas de atención
FF_DIM        = 256       # dimensión de la feed-forward
NUM_LAYERS    = 2         # capas encoder / decoder
DROPOUT       = 0.1

# ---------------- Tokenización ----------------
# Se añaden tokens especiales <start> y <end> a las respuestas
START_TOKEN = "<start>"
END_TOKEN   = "<end>"

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<oov>",
                      filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(train_in + [t for t in train_out])

word_index = tokenizer.word_index
VOCAB_REAL = min(VOCAB_SIZE, len(word_index) + 1)
print(f"Tamaño real del vocabulario: {VOCAB_REAL}")

def texts_to_seqs(texts, add_special=False):
    if add_special:
        texts = [f"{START_TOKEN} {t} {END_TOKEN}" for t in texts]
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

# Codificar entradas y respuestas
X_enc = texts_to_seqs(train_in, add_special=False)
Y_dec = texts_to_seqs(train_out, add_special=True)   # entrada del decoder (con <start>)
Y_tgt = texts_to_seqs(train_out, add_special=False)  # objetivo desplazado (sin <start>)
Y_tgt = np.array([np.append(seq[1:], 0) for seq in Y_dec])  # shift a la izquierda

X_test_enc = texts_to_seqs(test_in, add_special=False)

print(f"Forma X_enc: {X_enc.shape}, Y_dec: {Y_dec.shape}, Y_tgt: {Y_tgt.shape}")

In [ ]:
# ---------------- Codificación posicional ----------------
class PositionalEncoding(Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        pos = np.arange(max_len)[:, None]
        i = np.arange(d_model)[None, :]
        angle = pos / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle[:, 0::2] = np.sin(angle[:, 0::2])
        angle[:, 1::2] = np.cos(angle[:, 1::2])
        self.pos = tf.cast(angle[None], tf.float32)

    def call(self, x):
        return x + self.pos[:, :tf.shape(x)[1], :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({})
        return cfg

# ---------------- Capa Encoder ----------------
def encoder_layer(d_model, num_heads, ff_dim, dropout, name=None):
    inputs = Input(shape=(None, d_model))
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attn)
    ff = Dense(ff_dim, activation='relu')(out1)
    ff = Dense(d_model)(ff)
    ff = Dropout(dropout)(ff)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ff)
    return tf.keras.Model(inputs, out2, name=name)

# ---------------- Capa Decoder ----------------
def decoder_layer(d_model, num_heads, ff_dim, dropout, name=None):
    enc_in = Input(shape=(None, d_model), name=f'{name}_enc' if name else 'enc_in')
    dec_in = Input(shape=(None, d_model), name=f'{name}_dec' if name else 'dec_in')
    # Auto-atención causal
    attn1 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(dec_in, dec_in, use_causal_mask=True)
    attn1 = Dropout(dropout)(attn1)
    out1 = LayerNormalization(epsilon=1e-6)(dec_in + attn1)
    # Atención cruzada sobre el encoder
    attn2 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(out1, enc_in)
    attn2 = Dropout(dropout)(attn2)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + attn2)
    ff = Dense(ff_dim, activation='relu')(out2)
    ff = Dense(d_model)(ff)
    ff = Dropout(dropout)(ff)
    out3 = LayerNormalization(epsilon=1e-6)(out2 + ff)
    return tf.keras.Model([enc_in, dec_in], out3, name=name)

print("Capas Transformer definidas.")

In [ ]:
# ---------------- Modelo completo ----------------
def build_transformer(vocab_size, max_len, d_model, num_heads, ff_dim,
                      num_layers, dropout=0.1):
    enc_inputs = Input(shape=(max_len,), name='enc_inputs')
    dec_inputs = Input(shape=(max_len,), name='dec_inputs')

    enc_emb = Embedding(vocab_size, d_model, name='enc_embedding')(enc_inputs)
    enc_emb = PositionalEncoding(max_len, d_model)(enc_emb)
    x = enc_emb
    for i in range(num_layers):
        x = encoder_layer(d_model, num_heads, ff_dim, dropout, name=f'enc_{i}')(x)
    enc_out = x

    dec_emb = Embedding(vocab_size, d_model, name='dec_embedding')(dec_inputs)
    dec_emb = PositionalEncoding(max_len, d_model)(dec_emb)
    y = dec_emb
    for i in range(num_layers):
        y = decoder_layer(d_model, num_heads, ff_dim, dropout, name=f'dec_{i}')(dec_out, y)

    outputs = Dense(vocab_size, activation='softmax')(y)
    return tf.keras.Model([enc_inputs, dec_inputs], outputs, name='transformer_chatbot')

model = build_transformer(VOCAB_REAL, MAX_LEN, EMBED_DIM, NUM_HEADS, FF_DIM,
                          NUM_LAYERS, DROPOUT)
model.summary()

## 3. Entrenamiento del Modelo

Se utiliza **teacher forcing**: el decoder recibe como entrada la respuesta desplazada un token a la derecha (`Y_dec`) y se compara contra el objetivo (`Y_tgt`). La pérdida es `sparse_categorical_crossentropy`.

In [ ]:
BATCH_SIZE   = 64
EPOCHS       = 10
LEARNING_RATE = 0.001

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Para que el entrenamiento sea viable en Colab se usa una porción del conjunto
N_TRAIN = min(20000, len(X_enc))
history = model.fit(
    [X_enc[:N_TRAIN], Y_dec[:N_TRAIN]],
    Y_tgt[:N_TRAIN],
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el entrenamiento')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión durante el entrenamiento')
plt.xlabel('Época'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4. Evaluación del Modelo

Se mide la calidad de las respuestas generadas con la métrica **BLEU** y se incluye una función para conversar con el chatbot (generación autoregresiva token a token).

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

nltk.download('punkt', quiet=True)

index_word = {i: w for w, i in tokenizer.word_index.items() if i < VOCAB_REAL}

def secuencia_a_texto(seq):
    tokens = [index_word.get(int(i), '') for i in seq if int(i) != 0]
    return ' '.join(tokens).replace(START_TOKEN, '').replace(END_TOKEN, '').strip()

# ---------------- Generación autoregresiva (chatbot) ----------------
def responder(mensaje, max_new_tokens=MAX_LEN):
    enc = texts_to_seqs([mensaje], add_special=False)
    # El decoder arranca con el token <start>
    start_id = tokenizer.word_index.get(START_TOKEN, 1)
    end_id   = tokenizer.word_index.get(END_TOKEN, 2)
    dec = np.zeros((1, MAX_LEN), dtype='int32')
    dec[0, 0] = start_id

    for t in range(1, MAX_LEN):
        preds = model.predict([enc, dec], verbose=0)[0, t-1]
        next_id = int(np.argmax(preds))
        dec[0, t] = next_id
        if next_id == end_id:
            break
    return secuencia_a_texto(dec[0])

# Probar el chatbot con algunos mensajes
print("=== PRUEBA DEL CHATBOT ===\n")
for msg in ["How are you?", "What do you like to do?", "Let's go to the gym."]:
    print(f"Tú: {msg}")
    print(f"Bot: {responder(msg)}\n")

In [ ]:
# ---------------- Evaluación con BLEU ----------------
smoothie = SmoothingFunction().method1

N_EVAL = 200  # número de pares a evaluar (mantiene el tiempo razonable en Colab)
bleu_scores = []

for i in range(N_EVAL):
    ref = test_out[i].split()
    hip = responder(test_in[i]).split()
    if len(hip) == 0:
        continue
    score = sentence_bleu([ref], hip, smoothing_function=smoothie)
    bleu_scores.append(score)

print(f"BLEU promedio sobre {len(bleu_scores)} muestras: {np.mean(bleu_scores):.4f}")

# Mostrar algunos ejemplos de respuestas
print("\n--- Ejemplos de respuestas generadas ---")
for i in range(5):
    print(f"IN  : {test_in[i]}")
    print(f"REF : {test_out[i]}")
    print(f"BOT : {responder(test_in[i])}\n")

## 5. Ajuste de Hiperparámetros

Se prueban diferentes configuraciones de `num_heads` y `ff_dim` y se compara el rendimiento mediante la pérdida de validación. El modelo con menor pérdida se reporta como el mejor.

In [ ]:
resultados = []

for n_heads in [2, 4, 8]:
    for ff in [128, 256]:
        print(f"\n>>> Entrenando con num_heads={n_heads}, ff_dim={ff}")
        m = build_transformer(VOCAB_REAL, MAX_LEN, EMBED_DIM, n_heads, ff,
                               NUM_LAYERS, DROPOUT)
        m.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        h = m.fit([X_enc[:N_TRAIN], Y_dec[:N_TRAIN]], Y_tgt[:N_TRAIN],
                  batch_size=BATCH_SIZE, epochs=3, validation_split=0.1,
                  verbose=0)
        val_loss = h.history['val_loss'][-1]
        resultados.append({'num_heads': n_heads, 'ff_dim': ff, 'val_loss': val_loss})
        print(f"    val_loss final = {val_loss:.4f}")

df_resultados = pd.DataFrame(resultados).sort_values('val_loss')
print("\n=== RESUMEN DE AJUSTE DE HIPERPARÁMETROS ===")
print(df_resultados.to_string(index=False))

In [ ]:
import seaborn as sns

pivot = df_resultados.pivot(index='num_heads', columns='ff_dim', values='val_loss')
plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='viridis_r')
plt.title('Pérdida de validación por configuración')
plt.show()

mejor = df_resultados.iloc[0]
print(f"\nMejor configuración: num_heads={int(mejor['num_heads'])}, "
      f"ff_dim={int(mejor['ff_dim'])}, val_loss={mejor['val_loss']:.4f}")

## 6. Presentación de Resultados y Conclusiones

- **Resultados finales:** el modelo Transformer encoder-decoder logra generar respuestas coherentes a partir de los diálogos cotidianos de DailyDialog. La métrica **BLEU** cuantifica la similitud entre la respuesta generada y la de referencia.
- **Ajuste de hiperparámetros:** se compararon configuraciones de `num_heads` (2, 4, 8) y `ff_dim` (128, 256). La tabla y el heatmap del punto 5 muestran qué combinación produce menor pérdida de validación.
- **Conclusiones:**
  - La atención multi-cabezal permite capturar dependencias a largo plazo en el diálogo.
  - El encoder procesa el mensaje de entrada y el decoder genera la respuesta token a token con máscara causal.
  - El desempeño mejora al aumentar moderadamente la capacidad (más cabezas/ff_dim), pero con mayor costo computacional.
  - Limitaciones: el vocabulario y la longitud de secuencia se restringen para que el entrenamiento sea viable en Colab; un modelo más grande y más épocas mejorarían la fluidez.

¡Gracias por revisar el proyecto!